# 🧪 W3-D7 概念实验：第三周全景回顾

> 配套阅读：`ima/第3周-Day7-第三周总复习.md`
>
> 把预训练→SFT→RLHF→DPO→超参调优串成完整链路。

## 实验 1：五阶段训练流水线效果累积

每一步提升不同维度：预训练=语言+知识，SFT=指令，RLHF/DPO=安全，部署=速度。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
stages = ['原始\n数据','预训练','SFT','RLHF/DPO','部署\n优化']
scores = {'语言能力':[0,70,80,85,85],'知识储备':[0,75,76,76,76],
          '指令遵循':[0,10,85,88,90],'安全/友好':[0,5,30,85,90],'推理速度':[0,50,50,50,80]}
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(stages))
colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']
for i,(dim,vals) in enumerate(scores.items()):
    ax.plot(x, vals, '-o', color=colors[i], lw=2, ms=7, label=dim)
ax.set_xticks(x); ax.set_xticklabels(stages, fontsize=11)
ax.set_ylabel('能力分数（模拟示意）')
ax.set_title('大模型五阶段：每一步提升不同维度')
ax.legend(fontsize=10, loc='upper left'); ax.grid(True, alpha=0.3); ax.set_ylim(-5,100)
plt.tight_layout(); plt.show()
print("预训练→语言+知识；SFT→指令；RLHF/DPO→安全；部署→速度。")

## 实验 2：14 个核心知识点全景

W1-W3 共 14 个知识点，按难度 vs 重要性散点分布。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
concepts = ['自注意力','多头注意力','Transformer','位置编码','GPT架构',
            'FFN+残差','Tokenizer','KV Cache','Flash Attn','预训练','SFT','RLHF','DPO','超参调优']
diff = [8,7,7,6,5,5,4,6,8,5,4,8,7,6]
imp = [9,8,8,7,9,6,7,8,7,8,9,8,8,9]
fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(diff, imp, c=range(len(concepts)), cmap='tab20', s=150, edgecolors='black', alpha=0.85, zorder=5)
for i, name in enumerate(concepts):
    ax.annotate(name, (diff[i], imp[i]), textcoords='offset points', xytext=(6,6), fontsize=9)
ax.set_xlabel('理解难度（1-10）'); ax.set_ylabel('工程重要性（1-10）')
ax.set_title('W1-W3 核心知识点：右上角既重要又难')
ax.grid(True, alpha=0.3); ax.set_xlim(2.5,10); ax.set_ylim(5,10.5); plt.tight_layout(); plt.show()

## 实验 3：完整数据流模拟

用户输入→Tokenizer→Embedding→Attention→概率分布→采样输出。

In [ ]:
import numpy as np

text = "糖水店的招牌是"
print(f"输入: {text}"); print("="*50)

tokens = list(text)
print(f"Step1 Tokenizer: {tokens} ({len(tokens)} tokens)")

np.random.seed(9); embed_dim = 8
emb = {ch: np.random.randn(embed_dim)*0.1 for ch in set(tokens)}
vectors = np.stack([emb[t] for t in tokens])
print(f"Step2 Embedding: {vectors.shape} 矩阵")

np.random.seed(10)
attn = np.exp(np.random.randn(len(tokens))); attn /= attn.sum()
hidden = attn @ vectors
print(f"Step3 Attention: 权重 {np.round(attn,2).tolist()} → 隐向量 dim={embed_dim}")

np.random.seed(11)
candidates = ['红豆','绿豆','芒果','杨枝']
logits = np.random.randn(len(candidates))
probs = np.exp(logits) / np.exp(logits).sum()
chosen = candidates[np.argmax(probs)]
print(f"Step4 概率分布: {dict(zip(candidates, np.round(probs,3)))}")
print(f"Step5 采样输出: {chosen}")
print(f"\n最终: {text}{chosen}")

## 实验 4：各阶段超参数速查表

用热力图展示预训练/SFT/DPO 各阶段的超参数推荐值。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
params = ['学习率','Batch Size','Epochs','Warmup','LR Schedule','Weight Decay']
stages = ['预训练','SFT','DPO']
# 推荐值（归一化到 0-1 表示相对大小）
data = np.array([
    [0.5, 1.0, 0.0, 0.3, 0.8, 0.8],   # 预训练
    [0.2, 0.15, 0.3, 0.5, 0.8, 0.5],   # SFT
    [0.1, 0.15, 0.15, 0.3, 0.4, 0.2],  # DPO
])
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(data, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(params))); ax.set_xticklabels(params, fontsize=11)
ax.set_yticks(range(len(stages))); ax.set_yticklabels(stages, fontsize=12)
for i in range(len(stages)):
    for j in range(len(params)):
        val_labels = [['1e-4~3e-4','1024~4096','用steps','2-5%','Cosine','0.1'],
                      ['1e-5~5e-5','8~32','3~5','3-10%','Cosine','0.01~0.1'],
                      ['5e-7~5e-6','8~32','1~3','100步','Linear','0~0.01']]
        ax.text(j, i, val_labels[i][j], ha='center', va='center', fontsize=8,
                color='black' if data[i,j] < 0.7 else 'white')
ax.set_title('各阶段超参数速查表')
plt.tight_layout(); plt.show()

## 结论

| 问题 | 实验证据 |
|---|---|
| 五阶段分工 | 实验1：每步提升不同维度 |
| 知识点全景 | 实验2：14 个点，右上角重点掌握 |
| 完整数据流 | 实验3：Tokenizer→Emb→Attn→采样 |
| 超参速查 | 实验4：预训练/SFT/DPO 推荐值 |

→ 配套阅读：`ima/第3周-Day7-第三周总复习.md`